## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$? **The estimated price coefficient is -0.03766765298429363.**
2. Is it negative? Why is that important? **It is negative. That is important because the interpretation is that a 1 unit increase in price leads to about 3.7% decrease in brand share.**
3. Which product features are associated with higher demand? **Window share, compact share, and oven_style share all have positive coefficients, meaning they are positively associated with higher demand. Specifically window share and compact share have a coefficient of 12.88 and 9.81 respectively, which means a 1 unit increase in price leads to a 12.88% and 9.81% increase in brand share respectively. Winow and compat are the most associated with higher demand.**
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand. **Cuisineart, Ninja, and instant pot have the highest relative coefficients. They have higher brand share than the omitted brand holding everything else constant.**
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year. **None of them are particularly that much higher than the omitted variable, however, 2020 has the highest relative coefficient. Meaning 2020 captured the highest brand share relative to the omitted variable.**
6. What is the model's $R^2$? **The $R^2$ is 0.7634539500914361. This means that this linear model explains about 76% of the variation in the data. This is relatively strong, but it may be as a result of including brand and year dummies which have pretty strong correlation with brand share to begin with.**

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [8]:
df = pd.read_csv("air_fryers_clean_brand_year.csv")
df.head()

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364


In [9]:
from sklearn.linear_model import LinearRegression

brand_dummies = pd.get_dummies(df["brand"], drop_first = True)
year_dummies = pd.get_dummies(df["year"], drop_first = True)


X = pd.concat([df[["avg_price", "avg_rating", "compact_share", "dual_basket_share",
       "oven_style_share", "rotisserie_share", "window_share"]], brand_dummies, year_dummies], axis = 1)
y = df["log_brand_share"]
X.columns = X.columns.astype(str)

model = LinearRegression()
model.fit(X, y)
b_price = model.coef_[X.columns.get_loc("avg_price")]
b_price

np.float64(-0.03766765298429363)

In [10]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
})
coef_df

,feature,coefficient
0,avg_price,-0.037668
1,avg_rating,0.287517
2,compact_share,9.815304
3,dual_basket_share,-9.509686
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
6,window_share,12.880298
7,cosori,2.551946
8,cuisinart,6.422436
9,dash,0.176655


In [11]:
# Number 6
model.score(X, y)

0.7634539500914361